# Proyek Akhir Praktikum Big Data: TidarMart Data Platform
**Nama:** Wildan Maulana Habibi  
**NPM:** 2505060022  
**Seed:** 60022  
**Varian Pertanyaan Bisnis:** 2 (Fokus Produk dan Diskon)  

---

## Tahap 1: Penyimpanan Terdistribusi dan Ingesti (HDFS)

### Tujuan
1. Menyiapkan struktur direktori penyimpanan terdistribusi pada HDFS (`/tidarmart/raw`, `/tidarmart/stream_input`, `/tidarmart/curated`, dan `/tidarmart/models`).
2. Mengunggah seluruh hasil generator dataset (`transaksi`, `pelanggan.json`, `produk.json`, dan `stream_input`) dari folder lokal `data_proyek/` ke HDFS.
3. Memverifikasi ukuran berkas, jumlah blok penyimpanan, serta faktor replikasi di HDFS (`hdfs dfs -du -h` dan `hdfs fsck`).
4. Memastikan PySpark dapat membaca seluruh dataset langsung dari protokol `hdfs://localhost:9000/...` sesuai batasan memori VM.

In [2]:
#Buat struktur direktori di HDFS
!hdfs dfs -mkdir -p /tidarmart/raw/transaksi
!hdfs dfs -mkdir -p /tidarmart/raw/pelanggan
!hdfs dfs -mkdir -p /tidarmart/raw/produk
!hdfs dfs -mkdir -p /tidarmart/stream_input
!hdfs dfs -mkdir -p /tidarmart/curated
!hdfs dfs -mkdir -p /tidarmart/models

#Unggah dataset dari ./data_proyek/ ke masing-masing folder HDFS
!hdfs dfs -put -f data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
!hdfs dfs -put -f data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
!hdfs dfs -put -f data_proyek/produk.json /tidarmart/raw/produk/
!hdfs dfs -put -f data_proyek/stream_input/*.csv /tidarmart/stream_input/

print("Seluruh dataset berhasil diunggah ke HDFS.")

Seluruh dataset berhasil diunggah ke HDFS.


In [3]:
print("=== UKURAN BERKAS DI HDFS ===")
!hdfs dfs -du -h /tidarmart/raw
!hdfs dfs -du -h /tidarmart/stream_input

print("\n=== JUMLAH BLOK & FAKTOR REPLIKASI ===")
!hdfs fsck /tidarmart/raw -files -blocks

=== UKURAN BERKAS DI HDFS ===
6.0 M    6.0 M    /tidarmart/raw/pelanggan
680.7 K  680.7 K  /tidarmart/raw/produk
187.2 M  187.2 M  /tidarmart/raw/transaksi
479.7 K  479.7 K  /tidarmart/stream_input/stream_01.csv
479.7 K  479.7 K  /tidarmart/stream_input/stream_02.csv
479.3 K  479.3 K  /tidarmart/stream_input/stream_03.csv
479.1 K  479.1 K  /tidarmart/stream_input/stream_04.csv
479.7 K  479.7 K  /tidarmart/stream_input/stream_05.csv
479.6 K  479.6 K  /tidarmart/stream_input/stream_06.csv
479.2 K  479.2 K  /tidarmart/stream_input/stream_07.csv
478.9 K  478.9 K  /tidarmart/stream_input/stream_08.csv
479.4 K  479.4 K  /tidarmart/stream_input/stream_09.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_10.csv
479.4 K  479.4 K  /tidarmart/stream_input/stream_11.csv
479.6 K  479.6 K  /tidarmart/stream_input/stream_12.csv
479.8 K  479.8 K  /tidarmart/stream_input/stream_13.csv
479.2 K  479.2 K  /tidarmart/stream_input/stream_14.csv
479.8 K  479.8 K  /tidarmart/stream_input/stream_15.csv
479.

In [6]:
from pyspark.sql import SparkSession

# Inisialisasi SparkSession 
spark = (SparkSession.builder
    .appName('TidarMart')
    .master('local[2]')
    .config('spark.driver.memory', '1536m')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate())

# Membaca seluruh data dari HDFS 
df_transaksi = spark.read.csv("hdfs://localhost:9000/tidarmart/raw/transaksi/*.csv", header=True)
df_pelanggan = spark.read.json("hdfs://localhost:9000/tidarmart/raw/pelanggan/pelanggan.json")
df_produk    = spark.read.json("hdfs://localhost:9000/tidarmart/raw/produk/produk.json")

# Verifikasi jumlah baris
print(f"Transaksi terbaca : {df_transaksi.count():,} baris")
print(f"Pelanggan terbaca : {df_pelanggan.count():,} dokumen")
print(f"Produk terbaca    : {df_produk.count():,} dokumen")

Transaksi terbaca : 2,010,000 baris
Pelanggan terbaca : 50,000 dokumen
Produk terbaca    : 5,000 dokumen


### Jawaban Pertanyaan Konseptual & Interpretasi Tahap 1

**1. Mengapa direktori data mentah (`raw`) dan data bersih (`curated`) dipisah?**
* **Menjaga Keaslian Data (*Immutability*):** Direktori `/tidarmart/raw` berfungsi sebagai sumber kebenaran asli (*single source of truth*) dari hasil ingesti[cite: 4]. Apabila terjadi kesalahan logika saat proses pembersihan (ETL) atau perubahan aturan bisnis di kemudian hari, proses pembersihan dapat diulang dari awal tanpa merusak atau kehilangan data mentah[cite: 4].
* **Efisiensi Komputasi dan Keamanan Akses:** Direktori `/tidarmart/curated` menyimpan data yang sudah bersih dan terformat dalam struktur kolumnar (Parquet)[cite: 4]. Tahap analitik maupun *machine learning* cukup membaca dari `curated` sehingga eksekusi lebih cepat dan hemat memori dibandingkan memproses ulang data kotor[cite: 2, 4].

**2. Apa arti faktor replikasi $= 1$ pada VM *pseudo-distributed* dibanding klaster nyata?**
* **Pada VM *Pseudo-Distributed*:** Seluruh layanan HDFS (*NameNode* dan *DataNode*) berjalan pada satu mesin virtual dan satu disk fisik yang sama[cite: 1, 4]. Mengatur faktor replikasi lebih dari 1 tidak memberikan perlindungan terhadap kegagalan perangkat keras, melainkan hanya menggandakan pemakaian ruang penyimpanan VM dan memperlambat proses tulis (*disk I/O*)[cite: 3, 4].
* **Pada Klaster Produksi Nyata (*Multi-Node*):** Faktor replikasi standarnya diatur menjadi **3** (setiap blok data disalin ke tiga server/rak yang berbeda)[cite: 4]. Hal ini menjamin **toleransi kesalahan (*fault tolerance*)** agar data tidak hilang jika ada server yang mati, serta mempercepat pembacaan paralel melalui prinsip **lokalitas data (*data locality*)**.

**Hasil Verifikasi:**  
Data yang diunggah ke HDFS telah berhasil dibaca oleh PySpark melalui jalur `hdfs://localhost:9000/tidarmart/...` dengan jumlah baris yang cocok sepenuhnya dengan keluaran generator (**2.010.000** baris transaksi, **50.000** pelanggan, dan **5.000** produk)[cite: 4, 8].